# Venues

This notebook shows work done to derive the complete list of all unique venues in the Sounds of San Anto data set.

## Contents

- Data Sources
- Load and Inspect Data
- Confirm Ven_Ev_Ar_Address is Subset of Venue_Artist_Event
- Compare Layer 1 Venues To Main Data Source
- Add ISO Date Column
- Add First and Last Appearance Dates for Venues
- Add Unique Venues Indexes
- Write Venues Data Frame to Excel 

## Data Sources

There were a number of different Excel sheets with information on venues. Sometimes there were confusing relationships between them, which did not become clear until the data analysis process that is presented in this notebook.

1. `Layer 1 Venues`. In the Excel master document originally used for the Sounds of San Anto web visualization project, there is a sheet called `Layer 1 Venues.` This sheet has 975 venues with some addresses and coordinates. When I first encountered this data, 313 of these venues had coordinates as well as addresses.
2. `Venue_Artist_Event`. This is the original master list of concerts. Each concert is listed with a date, a venue, an artist, and in some cases an event name and a list of all the artists performing at the event.
3. `Ven_Ev_Ar_Address`. This sheet contains concerts from 1970 to 2009, with any address and coordinate data available from `Layer 1 Venues` merged. It is therefore derived sheet that combines data from the first two sheets.
4. `venues_with_coordinates`. This is a revised version of `Layer 1 Venues`. It contains coordinate data for 965 of the 975 venues in the sheet, derived by a team Information Specialist using the R Tidyverse library. In the process, a few of the venue names were edited.

There are additionally a few sheets that are derived from the first three by various data analysis processes. These will be discussed as needed.

Further, there are different versions of the data, 

## Load And Inspect Data

The first task is to load the main data sources and inspect them, so I can see the data structures and column names.

In [142]:
import pandas as pd
import numpy as np
import datetime
from dateutil import parser
import os

In [143]:
sounds_data = pd.ExcelFile('input-data//product1_data_master_rev.xlsx')
sheets = sounds_data.sheet_names
sheets

['Ven_Ev_Ar_Address',
 'Venue_Artist_Event',
 'Layer 1 Venues',
 'artistsGenre',
 'genreIndex',
 'Sheet5',
 'artistsIndex',
 'Venues No Address',
 'venues_with_coordinates']

Next, I will load data from the specific sheets that have venue names. I'm using the revised master sheet that was used as a basis for the web application, rather than the original master.

In [144]:
layer_1_venues = pd.read_excel('input-data/product1_data_master_rev.xlsx', sheet_name='Layer 1 Venues')
venue_artist_event = pd.read_excel('input-data//product1_data_master_rev.xlsx', sheet_name='Venue_Artist_Event')
ven_ev_ar_address = pd.read_excel('input-data//product1_data_master_rev.xlsx', sheet_name='Ven_Ev_Ar_Address')

In [145]:
layer_1_venues.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Venue_Name  974 non-null    str    
 1   Longitude   964 non-null    object 
 2   Latitude    964 non-null    float64
 3   Address     974 non-null    str    
 4   Zip         916 non-null    float64
 5   City        912 non-null    str    
 6   State       972 non-null    str    
dtypes: float64(2), object(1), str(4)
memory usage: 53.4+ KB


In [146]:
layer_1_venues.head()

,Venue_Name,Longitude,Latitude,Address,Zip,City,State
0,1033 Club,-98.480763,29.436116,1033 Avenue B,78215.0,San Antonio,Texas
1,151 Saloon,-98.690386,29.464097,10619 Westover Hills Blvd,78215.0,San Antonio,Texas
2,1902 Nightclub,-98.478216,29.421548,1174 East Commerce St,78205.0,San Antonio,Texas
3,210 Kapone's,-98.478815,29.425150,1223 East Houston,78205.0,San Antonio,Texas
4,4th Quarter Sports Bar,-98.569389,29.524564,8779 Wurzbach Rd,78240.0,San Antonio,Texas


In this table, we can see venues are in alphabetical order in a column called `Venue_Name`. Note that we will be handling the geolocation data in a later phase of analysis. There are a total of 974 venues listed.

In [147]:
venue_artist_event.info()

<class 'pandas.DataFrame'>
RangeIndex: 37892 entries, 0 to 37891
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   Month           37892 non-null  str  
 1   Day             37892 non-null  int64
 2   Year            37892 non-null  int64
 3   Venue           37892 non-null  str  
 4   Event_Artists   37892 non-null  str  
 5   Artist_Formula  37892 non-null  str  
 6   Event_Formulas  7723 non-null   str  
 7   Index           37892 non-null  int64
dtypes: int64(3), str(5)
memory usage: 2.3 MB


In [148]:
venue_artist_event.head()

,Month,Day,Year,Venue,Event_Artists,Artist_Formula,Event_Formulas,Index
0,September,10,1857,St. Mary's Catholic Church,French Mountaineer Singers,French Mountaineer Singers,NaN,1
1,June,25,1869,Menger Hotel,General Mason Farewell Party: San Antonio Bras...,San Antonio Brass Band,General Mason Farewell Party,2
2,July,4,1874,San Pedro Park,San Antonio Brass Band,San Antonio Brass Band,NaN,3
3,October,9,1874,"San Pedro Springs, Turner Hall",Tenth Texas Saengerfest,Tenth Texas Saengerfest,NaN,4
4,October,10,1874,"Casino Hall, Turner Hall",Tenth Texas Saengerfest,Tenth Texas Saengerfest,NaN,5


In [149]:
venue_artist_event.tail()

,Month,Day,Year,Venue,Event_Artists,Artist_Formula,Event_Formulas,Index
37887,November,30,2023,Paper Tiger,"Jeff Rosenstock, Small Crush",Small Crush,NaN,21993
37888,December,8,2023,Aztec Theater,Lil Darkie and the Collapse,Lil Darkie and the Collapse,NaN,21994
37889,December,9,2023,Buena Vista Theater,Walter Beasley,Walter Beasley,NaN,21995
37890,December,16,2023,Stetson Bar and Dance Hall,Small Town Habit,Small Town Habit,NaN,21996
37891,December,17,2023,Aztec Theater,Parliament Funkadelic featuring George Clinton,Parliament Funkadelic featuring George Clinton,NaN,21997


Next, I examine `venue_artist_event`. This is the main source of concert data for the full range of years from 1857 to 2023. There are 37892 concerts listed. Venues are listed in a column called `Venue`. There is a column called `Index`. This appears to be connected to *events* rather than concerts, where some events consisted of many concerts.

Finally, I'll examine `ven_ev_ar_address`. Hypothetically, there should not be any venues listed here that are not listed in `venue_artist_event`, since the former is derived from the latter.

In [150]:
ven_ev_ar_address.info()

<class 'pandas.DataFrame'>
RangeIndex: 20638 entries, 0 to 20637
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Index          20638 non-null  int64  
 1   Month          20638 non-null  str    
 2   Day            20638 non-null  int64  
 3   Year           20638 non-null  int64  
 4   Venue          20638 non-null  str    
 5   Event_Artists  20638 non-null  str    
 6   Address        17542 non-null  str    
 7   Zip            17540 non-null  float64
 8   Longitude      6446 non-null   float64
 9   Latitude       6446 non-null   float64
 10  Artist         20638 non-null  str    
 11  Event          4503 non-null   str    
 12  City           17540 non-null  str    
 13  State          17542 non-null  str    
dtypes: float64(3), int64(3), str(8)
memory usage: 2.2 MB


In [151]:
ven_ev_ar_address.head()

,Index,Month,Day,Year,Venue,Event_Artists,Address,Zip,Longitude,Latitude,Artist,Event,City,State
0,2581,January,1,1970,San Antonio Convention Center Arena,"KBER Holiday Show: Willie Nelson, Charley Pri...",601 Hemisfair Plaza Way,78205.0,NaN,NaN,Willie Nelson,KBER Holiday Show,San Antonio,Texas
1,2581,January,1,1970,San Antonio Convention Center Arena,"KBER Holiday Show: Willie Nelson, Charley Pri...",601 Hemisfair Plaza Way,78205.0,NaN,NaN,Charley Pride,KBER Holiday Show,San Antonio,Texas
2,2581,January,1,1970,San Antonio Convention Center Arena,"KBER Holiday Show: Willie Nelson, Charley Pri...",601 Hemisfair Plaza Way,78205.0,NaN,NaN,Grandpa Jones,KBER Holiday Show,San Antonio,Texas
3,2581,January,1,1970,San Antonio Convention Center Arena,"KBER Holiday Show: Willie Nelson, Charley Pri...",601 Hemisfair Plaza Way,78205.0,NaN,NaN,David Houston,KBER Holiday Show,San Antonio,Texas
4,2582,January,16,1970,San Antonio Municipal Auditorium,Vikki Carr,100 Auditorium Cir.,78205.0,NaN,NaN,Vikki Carr,NaN,San Antonio,Texas


## Confirm Ven_Ev_Ar_Address is Subset of Venue_Artist_Event

Next, I'll get the unique venues lists from the `Venue_Artist_Event` and `Ven_Ev_Ar_Address` and test my suspicion that the latter is a subset of the former. In the source data I find 2602 unique venues. Then I get the list of unique venues in the derived sheet. I concatenate the two and get the unique list from the concatenated list. This also yields 2602 venues, which confirms that `Ven_Ev_Ar_Address` has no unique venues that are not also in `Venue_Artist_Event`. We can therefore leave it out and work just with `Venue_Artist_Event`. 

In [152]:
vae_venues = venue_artist_event['Venue'].dropna().str.strip().unique()
vae_venues

<StringArray>
[                 'St. Mary's Catholic Church',
                                'Menger Hotel',
                              'San Pedro Park',
              'San Pedro Springs, Turner Hall',
                    'Casino Hall, Turner Hall',
 'San Pedro Springs, Alamo Plaza, Casino Hall',
                                 'Travis Park',
                                 'Casino Hall',
                                 'Turner Hall',
                      'San Pedro Springs Park',
 ...
                        'Tucker's Kozy Corner',
                           'Thirsty Camel Bar',
               'Brooster's Back Yard Icehouse',
                   'Koenig Park (Castroville)',
                 'University Methodist Church',
                         '500 Carolina Street',
                                  'R&J Saloon',
                                 'The Mansion',
          'Lonestar Premium Outdoors (Cibolo)',
                                    'Hangar 9']
Length: 2602, dtype: 

In [153]:
veaa_venues = ven_ev_ar_address['Venue'].dropna().str.strip().unique()
veaa_venues

<StringArray>
['San Antonio Convention Center Arena',    'San Antonio Municipal Auditorium',
                    'Freeman Coliseum',                             'Randy's',
                   'Farmer's Daughter',             'Blossom Athletic Center',
               'Sunken Garden Theater',                  'Trinity University',
                  'Lanier High School',                 'Ruth Taylor Theater',
 ...
                'Valentino's di Olmos',         'Wetmore Smokehouse & Saloon',
             'Get Up Community Center',                 'Hot Shots Billiards',
                          'The Gatsby',              'First Round Sports Bar',
                             'Pedicab',    'Whitewater Amphitheater, Sattler',
        'St. Lawrence Catholci Church',                       'Christ Church']
Length: 1548, dtype: str

In [154]:
len(set(np.concatenate((vae_venues, veaa_venues))))

2602

## Compare Layer 1 Venues To Main Data Source

Next I will compare the venue data in `Layer 1 Venues` to the venue data in `Venue_Artist_Event`. I strongly suspect there will be venues in the former that are not in the latter. These will be venues that have no associated concert data, but may have location data. Because they may have location data, and the locations were in many cases difficult to find, I will retain them, even though they will never appear in any concert history visualization.

Since the lists are derived from a Pandas data frame, they are numpy arrays. I use the numpy `setdiff1d` method to get the difference between the two. Sure enough, there are 369 venues in `Layer 1 Venues` that are not in `Venue_Artist_Event`. 

In [155]:
layer_1_venue_list = layer_1_venues['Venue_Name'].dropna().str.strip().unique()
layer_1_venue_list

<StringArray>
[                                                                       '1033 Club',
                                                                       '151 Saloon',
                                                                   '1902 Nightclub',
                                                                     '210 Kapone's',
                                                           '4th Quarter Sports Bar',
                                                                          '502 Bar',
                                                                      'Abracadabra',
                                                                   'Acapulco Sam's',
                                                                          'Accents',
                                                                    'Acquarium III',
 ...
                                                      'St. Henry's Catholic Church',
                                              

In [156]:
diff = np.setdiff1d(layer_1_venue_list, vae_venues)

In [157]:
diff

array(['108 Noth Centre', '4th Quarter Sports Bar', 'Abracadabra',
       'Alamo Ice House', "Aldaco's Mexican Restaurant",
       "Aldo's Ristorante Italian", 'Antonio Ballroom',
       'Antonio Strad Violin', 'Apple Tree Lounge', 'Aquariaum Disco',
       'Aragon Blue Room', 'Artisan on Alamo Distillery',
       'Artpace San Antonio', 'Azuca Nuevo Latino Restaurant & Bar',
       'Bar 414', 'Bar America', 'Bar Rojo at The Grand Hyatt Hotel',
       "Barclay's Piano Bar", "Beto's Comida Latina",
       'Bier Garten Riverwalk', 'Big Apple Night Spot',
       'Big Band Fun Club', "Big Bob's Burgers", 'Big Texas Ice House',
       'Bihl Haus Arts', "Billy D's Club", 'Black Canteen',
       'Blind Tiger Comedy Club', 'Blue Note Lounge',
       'Blue Star Brewing Co.', "Bo's Rodeo",
       "Bohanan's Prime Steaks & Seafood",
       'Boiler House Texas Grill & Wine Garden', 'Bombay Bicycle Club',
       'Bone Club', 'Bonesshakers Tap House and Pizzeria', 'Boozehounds',
       'Botika Peruvi

In [158]:
len(diff)

369

This does raise a problem, as I was unaware of this difference throughout the application development problem. I relied on `Layer 1 Venues` as the source of my authoritative venues list, without realizing that there was a mismatch with the list of concert venues. Efforts to geolocate venues focused on the list in `Layer 1 Venues`, but as a result a lot of effort was put into locating venues that may not even have concerts associated, and so would not even appear on the concert map. In retrospect, it would have been better to derive the unique venue list from `Ven_Ev_Ar_Address` and focus on geolocating those. That is the direction future geolocation efforts should take.

I am curious to know how many venues are in `Ven_Ev_Ar_Address` but not in `Layer 1 Venues`. It looks like there are 1079 of these. This is where future geolocation work should focus.

In [159]:
not_in_layer_1 = np.setdiff1d(veaa_venues, layer_1_venue_list)

In [160]:
len(not_in_layer_1)

1079

## Add ISO Date Column

I'm ready to construct our unique venue list. As in the case of artists, I would like each venue to have a unique identifier. However, in this case there is no pre-existing index to maintain. 

As with artists, I would also like to be able to sort the venues by date of first and last appearance in the data. This will be possible only for the venues that are listed in `Venue_Artist_Event`. Those that are listed *only* in Layer 1 Venues will have no associated dates.

I'll use the same process as for artists to add ISO-formatted dates for all concerts in the data frame.

In [161]:
def str_date_to_iso(month, day, year):
    str_date = f"{month} {day} {year}"
    dt = parser.parse(str_date).date().isoformat()
    return dt

In [162]:
venue_artist_event["date"] = venue_artist_event.apply(lambda row: str_date_to_iso(row['Month'], row['Day'], row['Year']), axis=1)

In [163]:
venue_artist_event.head()

,Month,Day,Year,Venue,Event_Artists,Artist_Formula,Event_Formulas,Index,date
0,September,10,1857,St. Mary's Catholic Church,French Mountaineer Singers,French Mountaineer Singers,NaN,1,1857-09-10
1,June,25,1869,Menger Hotel,General Mason Farewell Party: San Antonio Bras...,San Antonio Brass Band,General Mason Farewell Party,2,1869-06-25
2,July,4,1874,San Pedro Park,San Antonio Brass Band,San Antonio Brass Band,NaN,3,1874-07-04
3,October,9,1874,"San Pedro Springs, Turner Hall",Tenth Texas Saengerfest,Tenth Texas Saengerfest,NaN,4,1874-10-09
4,October,10,1874,"Casino Hall, Turner Hall",Tenth Texas Saengerfest,Tenth Texas Saengerfest,NaN,5,1874-10-10


## Add First and Last Appearance Dates for Venues

Now I will repeat the process used for artists to create a data frame with the dates of first and last appearance for all venues.

In [164]:
venue_artist_event["Venue"] = venue_artist_event["Venue"].str.strip()

venue_dates = (
    venue_artist_event
    .groupby("Venue")["date"]
    .agg(["min", "max"])
    .reset_index()
)

venue_dates

,Venue,min,max
0,"""Menudo Acres""",1978-10-06,1978-10-07
1,1033 Club,1991-06-24,1991-09-28
2,110 Broadway Building Office,1985-07-12,1987-08-10
3,11th Street Cowboy Bar (Bandera),2007-07-14,2023-07-29
4,136 South Acme,2023-03-04,2023-03-04
...,...,...,...
2597,Zion Star Baptist Church,1954-03-14,1954-03-14
2598,Zombies,2009-10-03,2018-04-27
2599,the Korova,2011-08-30,2017-10-21
2600,the Solarium,2003-06-19,2003-06-19


In [165]:
venue_dates.info()

<class 'pandas.DataFrame'>
RangeIndex: 2602 entries, 0 to 2601
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Venue   2602 non-null   str  
 1   min     2602 non-null   str  
 2   max     2602 non-null   str  
dtypes: str(3)
memory usage: 61.1 KB


I see that for some reason, pandas read the dates in as strings rather than datetime objects, so I'll fix that.

In [166]:
venue_dates['min'] = pd.to_datetime(venue_dates['min'])
venue_dates['max'] = pd.to_datetime(venue_dates['max'])

Now I'll generate a data frame that includes all the venues in `Layer 1 Venues` that are not currently in `Venue_Artist_Event`. These are stored in a list with the variable name `diff`, generated earlier.

In [167]:
new_rows = pd.DataFrame({
    'Venue': diff,
    'min': pd.NaT,
    'max': pd.NaT
})

venue_dates = pd.concat([venue_dates, new_rows], ignore_index=True)

In [168]:
venue_dates

,Venue,min,max
0,"""Menudo Acres""",1978-10-06,1978-10-07
1,1033 Club,1991-06-24,1991-09-28
2,110 Broadway Building Office,1985-07-12,1987-08-10
3,11th Street Cowboy Bar (Bandera),2007-07-14,2023-07-29
4,136 South Acme,2023-03-04,2023-03-04
...,...,...,...
2966,Wyndham Garden Riverwalk,NaT,NaT
2967,Yellow Rose Country and Western Bar,NaT,NaT
2968,Zanzibar Nite Club,NaT,NaT
2969,Zip 2 Five Sports and Spirits,NaT,NaT


In [169]:
venue_dates.info()

<class 'pandas.DataFrame'>
RangeIndex: 2971 entries, 0 to 2970
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Venue   2971 non-null   str           
 1   min     2602 non-null   datetime64[ns]
 2   max     2602 non-null   datetime64[ns]
dtypes: datetime64[ns](2), str(1)
memory usage: 69.8 KB


## Add Unique Venues Indexes

Now, I can add a unique identifier for each venue. When developing the Sounds of San Anto concert history visualization, I assigned unique indexes to all the venues in `Layer 1 Venues`. I would like to retain those indexes. For the rest of the venues, I would like them to appear in chronological order, according to how early the appear in the concert history data.

Accordingly, first I'm going to sort the venues with dates by order of appearance in the historical data. 

In [170]:
venue_dates = venue_dates.sort_values('min', na_position='last').reset_index(drop=True)

In [171]:
venue_dates.head()

,Venue,min,max
0,St. Mary's Catholic Church,1857-09-10,1857-09-10
1,Menger Hotel,1869-06-25,1980-01-26
2,San Pedro Park,1874-07-04,1991-11-02
3,"San Pedro Springs, Turner Hall",1874-10-09,1874-10-09
4,"Casino Hall, Turner Hall",1874-10-10,1874-10-10


In [172]:
venue_dates.tail()

,Venue,min,max
2966,Wyndham Garden Riverwalk,NaT,NaT
2967,Yellow Rose Country and Western Bar,NaT,NaT
2968,Zanzibar Nite Club,NaT,NaT
2969,Zip 2 Five Sports and Spirits,NaT,NaT
2970,Zombies Bar and Live Music Venue,NaT,NaT


I'll rename 'min' and 'max' to have more expressive names that indicate what the columns actually contain.

In [173]:
venue_dates = venue_dates.rename(columns={'min': 'First Appearance', 'max': 'Last Appearance'})

Now, I'm going to import the version of the venues data that has unique ids. These are in a JSON file I used as a data source for the Sounds of San Anto data visualization.

In [175]:
venues_with_ids = pd.read_json('input-data/venues.json')
venues_with_ids.info()

<class 'pandas.DataFrame'>
RangeIndex: 974 entries, 0 to 973
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         974 non-null    int64  
 1   name       974 non-null    str    
 2   address    974 non-null    str    
 3   city       912 non-null    str    
 4   state      972 non-null    str    
 5   zip        916 non-null    float64
 6   longitude  903 non-null    float64
 7   latitude   903 non-null    float64
dtypes: float64(3), int64(1), str(4)
memory usage: 61.0 KB


I'll create a data frame that is basically a lookup table, with the name of each venue as the index, and its id as the associated value.

In [178]:
name_id_lookup = venues_with_ids.set_index('name')['id']
name_id_lookup.head()

name
1033 Club                 1
151 Saloon                2
1902 Nightclub            3
210 Kapone's              4
4th Quarter Sports Bar    5
Name: id, dtype: int64

Next, I'll create a new column in the `venue_dates` data frame, called "Index". I'll set values by using the `.map` method on the `Venue` column values and passing it our lookup data frame. This will look for any values of `Venue` that match index values in the lookup table, and assign the result to the new column.

As expected, we get 974 non-null values for the Index column.

In [179]:
venue_dates['Index'] = venue_dates['Venue'].map(name_id_lookup)

In [180]:
venue_dates.info()

<class 'pandas.DataFrame'>
RangeIndex: 2971 entries, 0 to 2970
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Venue             2971 non-null   str           
 1   First Appearance  2602 non-null   datetime64[ns]
 2   Last Appearance   2602 non-null   datetime64[ns]
 3   Index             974 non-null    float64       
dtypes: datetime64[ns](2), float64(1), str(1)
memory usage: 93.0 KB


Next, I'll create a boolean mask for the Index. This will have True values for all the rows that have no index yet. I'll calculate the total of null indexes. We have 1997 venues with no index at this point.

In [183]:
null_mask = venue_dates['Index'].isna()
total_missing = null_mask.sum()
total_missing

np.int64(1997)

Now, I will calculate where the new indexes need to start. This is will start at the currently highest index plus one, which is 975.

In [184]:
index_start = int(venue_dates['Index'].max()) + 1
index_start

975

Finally, I assign the new indexes. Here I am using row/column selection syntax to select all the values in the index column that have null indexes (`venue_dates.loc[null_mask, 'Index']`. Then I create a range object going from the new index start for as many as are missing, and use that to assign values wherever they are missing.

Finally, I convert the column to integer values. It started out as floating points because there were null values.

In [186]:
venue_dates.loc[null_mask, 'Index'] = range(index_start, index_start + total_missing)

In [188]:
venue_dates['Index'] = venue_dates['Index'].astype(int)

In [189]:
venue_dates

,Venue,First Appearance,Last Appearance,Index
0,St. Mary's Catholic Church,1857-09-10,1857-09-10,975
1,Menger Hotel,1869-06-25,1980-01-26,928
2,San Pedro Park,1874-07-04,1991-11-02,976
3,"San Pedro Springs, Turner Hall",1874-10-09,1874-10-09,977
4,"Casino Hall, Turner Hall",1874-10-10,1874-10-10,978
...,...,...,...,...
2966,Wyndham Garden Riverwalk,NaT,NaT,833
2967,Yellow Rose Country and Western Bar,NaT,NaT,834
2968,Zanzibar Nite Club,NaT,NaT,836
2969,Zip 2 Five Sports and Spirits,NaT,NaT,838


I'll move the index column over so it's the first column.

In [190]:
venue_dates = venue_dates[['Index', 'Venue', 'First Appearance', 'Last Appearance']]
venue_dates

,Index,Venue,First Appearance,Last Appearance
0,975,St. Mary's Catholic Church,1857-09-10,1857-09-10
1,928,Menger Hotel,1869-06-25,1980-01-26
2,976,San Pedro Park,1874-07-04,1991-11-02
3,977,"San Pedro Springs, Turner Hall",1874-10-09,1874-10-09
4,978,"Casino Hall, Turner Hall",1874-10-10,1874-10-10
...,...,...,...,...
2966,833,Wyndham Garden Riverwalk,NaT,NaT
2967,834,Yellow Rose Country and Western Bar,NaT,NaT
2968,836,Zanzibar Nite Club,NaT,NaT
2969,838,Zip 2 Five Sports and Spirits,NaT,NaT


## Write Venues Data Frame to Excel

Finally, I'll write the resulting data frame to our Excel sheet, using the method that I generated for artists. Exporting pandas datetimes to Excel can be very challenging, so I just convert to strings before exporting. 

In [191]:
def write_df_to_excel(df, filepath, sheet_name):
    """Write a DataFrame to a sheet in an Excel file, creating the file
    if it doesn't exist, or replacing the sheet if it does."""
    
    mode = "a" if os.path.exists(filepath) else "w"
    kwargs = {"if_sheet_exists": "replace"} if mode == "a" else {}

    with pd.ExcelWriter(filepath, engine="openpyxl", datetime_format='YYYY-MM-DD', mode=mode, **kwargs) as writer:
        df.to_excel(writer, sheet_name=sheet_name, index=False)

In [192]:
venue_dates_str = venue_dates
venue_dates_str['First Appearance'] = venue_dates_str['First Appearance'].dt.strftime('%Y-%m-%d')
venue_dates_str['Last Appearance'] = venue_dates_str['Last Appearance'].dt.strftime('%Y-%m-%d')

In [194]:
# write_df_to_excel(venue_dates_str, 'output-data/sounds-of-san-anto.xlsx', 'venues')